# Experiment 2: location extraction

Evaluates how well the VLM extracts structured location information from indoor
signage on the UFV campus dataset. It runs the two Experiment 1 finalists (the top
performer of each family: Gemma 4 E4B-it and PaliGemma2-3b-mix).

## Questions

| Question | Category | n | Type |
|-|-|-|-|
| `floor_extraction` | directory_signs | 20 (5 agora + 5 per floor 0, 1, 2) | Multi-class, 4 balanced classes |
| `room_code_extraction` | room_plates | 26 (codes like 1.6, 2.1, ai_lab) | Free-form extraction |

47 tasks per model.

## Metrics (per question, per model)

For `floor_extraction` (4-class balanced):
- `floor_accuracy`, `building_accuracy`, `location_accuracy`.
- Macro precision, recall and F1 over the floor classes.
- `bertscore_f1` on the full raw response.
- Confusion matrix and average output length.

For `room_code_extraction` (free-form):
- `room_code_accuracy`: substring match with normalised delimiters.
- Average output length.

Cross-model: overall accuracy across the 47 tasks, plus operational metrics (VRAM,
latency, throughput, load time).

## Inputs at `/content/`

- `images.zip`: campus images (same set as Experiment 1).
- `annotations.zip`: `evaluation_prompts.json` and `ground_truth.json`.
- `models.json`: copy of `config/models.json`.

## Outputs

```
/content/results/
  <model>/
    results.csv      one row per (image, question)
    aggregate.csv    per-question aggregate metrics
    efficiency.json  operational and aggregated metrics
    matrices.json    confusion matrices and per-class metrics
  all_models_summary.csv
```

## 1. Install dependencies

On a fresh runtime, run this cell, then Runtime -> Restart runtime, then continue
from section 2.

In [ ]:
!pip install -qU --force-reinstall Pillow
!pip install -qU transformers accelerate bert-score scipy
print()
print('=' * 60)
print('FIRST RUN ON FRESH RUNTIME: Runtime -> Restart runtime now,')
print('then run cells from sec 2 onwards (do NOT re-run sec 1).')
print('=' * 60)


## 2. Unzip dataset

Unpacks `images.zip` and `annotations.zip` into `/content/`. Idempotent.

In [ ]:
import os
import zipfile

CONTENT = '/content'
ARCHIVES = [
    ('images.zip', 'images'),
    ('annotations.zip', 'annotations'),
]

for zip_name, folder_name in ARCHIVES:
    zip_path = os.path.join(CONTENT, zip_name)
    dest = os.path.join(CONTENT, folder_name)
    if not os.path.isfile(zip_path):
        raise FileNotFoundError(f'Missing {zip_path!r}. Upload {zip_name} to /content/.')
    os.makedirs(dest, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(dest)
    print(f'OK: {zip_name} -> {dest}/')

for p in (
    os.path.join(CONTENT, 'annotations', 'evaluation_prompts.json'),
    os.path.join(CONTENT, 'annotations', 'ground_truth.json'),
):
    if not os.path.isfile(p):
        raise FileNotFoundError(f'Expected file missing after unzip: {p}')

for sub in ('directory_signs', 'room_plates'):
    d = os.path.join(CONTENT, 'images', sub)
    if os.path.isdir(d):
        n = len([f for f in os.listdir(d) if f.lower().endswith('.jpg')])
        print(f'  images/{sub}/: {n} .jpg files')
    else:
        print(f'  WARNING: missing folder {d}')

print('Unzip complete.')


## 3. Configuration

Paths, decoding protocol and seeds. Same greedy, deterministic protocol as
Experiment 1.

In [ ]:
import random
import numpy as np
import torch

HF_TOKEN = ''  # paste your HuggingFace read token here (PaliGemma and Gemma weights are gated)

IMAGES_PATH      = '/content/images'
ANNOTATIONS_PATH = '/content/annotations'
MODELS_JSON_PATH = '/content/models.json'
RESULTS_PATH     = '/content/results/exp2_update'

MAX_NEW_TOKENS = 30

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DECODING_PROTOCOL = {
    'do_sample': False,
    'num_beams': 1,
    'temperature': None,
    'max_new_tokens': MAX_NEW_TOKENS,
    'enable_thinking': False,
    'torch_dtype': 'auto on CUDA (bfloat16); float32 on CPU',
    'gemma_recommended_overridden': 'stochastic (T=1.0, top_p=0.95, top_k=64) replaced by greedy for reproducibility',
}

N_WARMUP = 3
N_LATENCY_RUNS = 3

os.makedirs(RESULTS_PATH, exist_ok=True)

print('Config OK')
print(f'Images:      {IMAGES_PATH}')
print(f'Annotations: {ANNOTATIONS_PATH}')
print(f'Results:     {RESULTS_PATH}')
print(f'MAX_NEW_TOKENS: {MAX_NEW_TOKENS}')
print(f'Seed: {SEED}  |  cudnn deterministic: True')
print(f'Latency protocol: N_WARMUP={N_WARMUP}, N_LATENCY_RUNS={N_LATENCY_RUNS}')


## 4. Reproducibility card

Written to `exp2_reproducibility.json`: library versions, GPU, decoding and latency
protocol, and SHA-256 hashes of the dataset artefacts.

In [ ]:
import subprocess
import datetime
import json
import hashlib
import transformers
from importlib import metadata as importlib_metadata


def get_gpu_name():
    if torch.cuda.is_available():
        return torch.cuda.get_device_name(0)
    return 'CPU'


def get_git_commit():
    try:
        return subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'],
                                       stderr=subprocess.DEVNULL).decode().strip()
    except Exception:
        return 'unavailable'


def _pkg_version(name):
    try:
        return importlib_metadata.version(name)
    except Exception:
        return 'unavailable'


def _file_sha256(path, chunk_size=65536):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()


with open(MODELS_JSON_PATH) as f:
    models_config = json.load(f)

run_models = models_config['models']

DATASET_ARTIFACT_PATHS = [
    '/content/images.zip',
    '/content/annotations/ground_truth.json',
    '/content/annotations/evaluation_prompts.json',
    '/content/models.json',
]
dataset_artifact_hashes = {}
for p in DATASET_ARTIFACT_PATHS:
    if os.path.isfile(p):
        dataset_artifact_hashes[p] = {
            'sha256': _file_sha256(p),
            'size_bytes': os.path.getsize(p),
        }

LATENCY_PROTOCOL = {
    'n_warmup': N_WARMUP,
    'n_latency_runs': N_LATENCY_RUNS,
    'cuda_synchronize_around_timer': True,
    'warmup_image_source': 'synthetic 224x224 deterministic RGB noise, seed=SEED',
    'image_order': 'shuffled deterministically per run (seed = SEED*1000 + run_idx)',
}

repro = {
    'experiment': 'exp2_update',
    'date': datetime.datetime.now().isoformat(timespec='seconds'),
    'gpu': get_gpu_name(),
    'torch_version': torch.__version__,
    'transformers_version': transformers.__version__,
    'pillow_version': _pkg_version('pillow'),
    'bert_score_version': _pkg_version('bert-score'),
    'seed': SEED,
    'cudnn_deterministic': True,
    'max_new_tokens': MAX_NEW_TOKENS,
    'images_path': IMAGES_PATH,
    'annotations_path': ANNOTATIONS_PATH,
    'git_commit': get_git_commit(),
    'models_evaluated': [m['model_id'] for m in run_models],
    'decoding_protocol': DECODING_PROTOCOL,
    'latency_protocol': LATENCY_PROTOCOL,
    'dataset_artifact_hashes': dataset_artifact_hashes,
}

repro_path = f'{RESULTS_PATH}/exp2_reproducibility.json'
with open(repro_path, 'w') as f:
    json.dump(repro, f, indent=2)

print('REPRODUCIBILITY CARD')
print('=' * 40)
preview = {k: v for k, v in repro.items() if k != 'dataset_artifact_hashes'}
preview['dataset_artifact_hashes'] = f'<{len(dataset_artifact_hashes)} SHA-256 fingerprints saved to JSON>'
for k, v in preview.items():
    print(f'  {k}: {v}')


## 5. Imports and dataset loading

Loads the Experiment 2 questions from `evaluation_prompts.json` and the per-image expected
fields from `ground_truth.json`.

In [ ]:
import gc
import re
import time
from collections import defaultdict
from PIL import Image
import pandas as pd
from transformers import AutoModelForCausalLM, AutoProcessor, PaliGemmaForConditionalGeneration
from bert_score import score as bertscore_fn

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 200)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

with open(os.path.join(ANNOTATIONS_PATH, 'evaluation_prompts.json')) as f:
    questions_payload = json.load(f)
with open(os.path.join(ANNOTATIONS_PATH, 'ground_truth.json')) as f:
    ground_truth = json.load(f)

exp2_questions = questions_payload['experiment2']['questions']

print(f'Exp2 questions: {len(exp2_questions)}')
for q in exp2_questions:
    print(f'  - {q["id"]}: applies_to={q["applies_to_categories"]}, task_type={q["task_type"]}')
print(f'Ground truth entries: {len(ground_truth)}')


## 6. Metric primitives

Extraction helpers, robust to formatting variation:

- `parse_directory_sign_response`: extracts the `(building, floor)` pair, handling `G 1`, `g1`, `Building G floor 1` and similar.
- `room_code_match`: substring match with delimiters `[._\-,]` normalised to spaces, so `1.6`, `1,6` and `Room 1.6` all match.
- Per-class P/R/F1, macro aggregation and the confusion matrix for the 4-class floor problem.
- BERTScore F1 for `floor_extraction` (raw response vs expected string).

In [ ]:
def normalize(s):
    """Lowercase, strip, drop punctuation, collapse whitespace. Handles None."""
    if s is None:
        return ''
    s = str(s).lower().strip()
    s = re.sub(r'[^\w\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


def parse_directory_sign_response(response):
    """Extract (building, floor_label) from any response variant.

    Returns tuple like ('G', '1') or ('G', 'agora'). Both can be None if not detected.
    Robust to: 'G 1', 'g1', 'G 1.', 'Building G floor 1', 'agora', 'G Agora', etc.
    """
    if response is None or not str(response).strip():
        return None, None
    norm = response.lower().strip()

    # Building detection: look for 'g' as standalone, or 'g' adjacent to a digit/agora
    building = None
    if re.search(r'\bg\b', norm) or re.search(r'\bg\s*[012agora]', norm) or re.search(r'\bg[012]\b', norm):
        building = 'G'
    elif re.search(r'\bagora\b', norm):
        building = 'G'  # agora is part of G in this dataset

    # Floor detection
    if re.search(r'\bagora\b', norm):
        floor = 'agora'
    else:
        # First try: 'g' followed by digit with optional whitespace (handles 'g0', 'g 0', 'G1', 'G 1')
        m = re.search(r'\bg\s*([012])\b', norm)
        if m:
            floor = m.group(1)
        else:
            # Fallback: standalone digit anywhere
            m = re.search(r'\b([012])\b', norm)
            floor = m.group(1) if m else None

    return building, floor


def format_parsed(building, floor):
    """Convert parsed tuple back to canonical 'G X' string for display/comparison."""
    if building is None or floor is None:
        return 'unparseable'
    if floor == 'agora':
        return f'{building} Agora'
    return f'{building} {floor}'


ROOM_CODE_ALIASES = {
    # Special non-numeric room codes get explicit alias lists
    'ai_lab': ['ai_lab', 'ai lab', 'ai laboratory', 'i.a', 'ia', 'lab ia', 'laboratorio ia',
               'lab de ia', 'inteligencia artificial', 'lab inteligencia artificial'],
}


def room_code_match(response, expected_code):
    """Lenient room-code match.

    For numeric codes (e.g. '3', '1.4', '2.1'): extract all numbers from the response
    (decimals + integers) with `re.findall(r'\d+\.\d+|\d+', response)`. Match if the
    expected number appears EXACTLY in the extracted list. Accepts 'AULA 3', 'AULA 3shot',
    'Room 1.4 is on second floor'. Rejects 'AULA 11' (different number), '14' (no dot,
    different number), 'ufv' (no digits).

    For non-numeric codes (e.g. 'ai_lab'): check a curated alias list with
    word-boundary regex. Accepts 'I.A', 'AI lab', etc.
    """
    if response is None or expected_code is None:
        return False

    norm_resp = str(response).lower().strip()
    expected_str = str(expected_code).lower().strip()

    # Numeric code path: digits or decimals only
    if re.match(r'^[\d.]+$', expected_str):
        nums = re.findall(r'\d+\.\d+|\d+', norm_resp)
        return expected_str in nums

    # Non-numeric code path: alias-based matching
    candidates = ROOM_CODE_ALIASES.get(expected_str, [expected_str])
    for cand in candidates:
        pattern = r'\b' + re.escape(cand.lower()) + r'\b'
        if re.search(pattern, norm_resp):
            return True
    return False


def accuracy(y_true, y_pred):
    if not y_true:
        return 0.0
    return sum(1 for t, p in zip(y_true, y_pred) if t == p) / len(y_true)


def per_class_prf(y_true, y_pred, classes):
    """Per-class precision/recall/F1 over the given classes."""
    out = {}
    for c in classes:
        tp = sum(1 for t, p in zip(y_true, y_pred) if t == c and p == c)
        fp = sum(1 for t, p in zip(y_true, y_pred) if t != c and p == c)
        fn = sum(1 for t, p in zip(y_true, y_pred) if t == c and p != c)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        out[c] = {
            'precision': round(precision, 4),
            'recall':    round(recall, 4),
            'f1':        round(f1, 4),
            'support':   sum(1 for t in y_true if t == c),
            'tp': tp, 'fp': fp, 'fn': fn,
        }
    return out


def macro_prf(y_true, y_pred, classes):
    per_c = per_class_prf(y_true, y_pred, classes)
    supported = [c for c in classes if per_c[c]['support'] > 0]
    if not supported:
        return {'macro_precision': 0.0, 'macro_recall': 0.0, 'macro_f1': 0.0, 'classes_used': []}
    p_macro = sum(per_c[c]['precision'] for c in supported) / len(supported)
    r_macro = sum(per_c[c]['recall'] for c in supported) / len(supported)
    f1_macro = sum(per_c[c]['f1'] for c in supported) / len(supported)
    return {
        'macro_precision': round(p_macro, 4),
        'macro_recall':    round(r_macro, 4),
        'macro_f1':        round(f1_macro, 4),
        'classes_used':    supported,
    }


def confusion_matrix_dict(y_true, y_pred, classes):
    """{true: {pred: count}} dict. Adds 'unparseable' / 'other' columns for unrecognized predictions."""
    cols = list(classes) + ['unparseable']
    mat = {c: {c2: 0 for c2 in cols} for c in classes}
    for t, p in zip(y_true, y_pred):
        if t not in mat:
            continue
        if p in mat[t]:
            mat[t][p] += 1
        else:
            mat[t]['unparseable'] += 1
    return mat


def avg_output_length_words(predictions):
    if not predictions:
        return 0.0
    return round(sum(len(str(p).split()) for p in predictions) / len(predictions), 2)


def compute_bertscore(predictions, references):
    """Mean BertScore F1 over (pred, ref) pairs. Skips empty predictions."""
    pairs = [(p, r) for p, r in zip(predictions, references) if p and r]
    if not pairs:
        return 0.0
    preds, refs = zip(*pairs)
    P, R, F1 = bertscore_fn(list(preds), list(refs), lang='en', verbose=False)
    return round(float(F1.mean()), 4)


print('Metric primitives defined')


## 7. Model loading (family dispatch, same as Experiment 1)

Dispatches on the `family` field in `models.json`, exactly as in Experiment 1.

In [ ]:
def make_synthetic_image(seed=None):
    rng = np.random.default_rng(SEED if seed is None else seed)
    arr = rng.integers(0, 256, size=(224, 224, 3), dtype=np.uint8)
    return Image.fromarray(arr, mode='RGB')


def load_model(meta):
    family = meta.get('family', 'paligemma')
    model_id = meta['model_id']

    if family == 'paligemma':
        dtype = torch.bfloat16 if DEVICE == 'cuda' else torch.float32
        processor = AutoProcessor.from_pretrained(model_id, token=HF_TOKEN)
        model = PaliGemmaForConditionalGeneration.from_pretrained(
            model_id, token=HF_TOKEN,
            torch_dtype=dtype,
            device_map='auto' if DEVICE == 'cuda' else None,
        )
        if DEVICE != 'cuda':
            model = model.to(DEVICE)
        model.eval()
        return processor, model, family

    elif family == 'gemma4':
        processor = AutoProcessor.from_pretrained(model_id, token=HF_TOKEN)
        model = AutoModelForCausalLM.from_pretrained(
            model_id, token=HF_TOKEN,
            dtype='auto' if DEVICE == 'cuda' else torch.float32,
            device_map='auto' if DEVICE == 'cuda' else None,
        )
        if DEVICE != 'cuda':
            model = model.to(DEVICE)
        model.eval()
        return processor, model, family

    else:
        raise ValueError(f'Unknown family {family!r} for model {model_id}')


print('Model loader defined')


## 8. Inference primitives (same dispatch as Experiment 1)

`run_inference_timed` handles both model families with greedy decoding and
CUDA-synchronised timing.

In [ ]:
GEMMA_SYSTEM_PROMPT = (
    'You are a visual assistant for indoor navigation. '
    'Answer concisely and exactly in the format requested by the question. '
    'Do not add explanations or extra words.'
)


def _sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def build_paligemma_inputs(processor, image, question, model):
    prompt = f'<image>{question}'
    inputs = processor(text=prompt, images=image, padding='longest',
                       do_convert_rgb=True, return_tensors='pt').to(model.device)
    inputs = inputs.to(dtype=model.dtype)
    return inputs


def build_gemma_inputs(processor, image, question, model):
    messages = [
        {'role': 'system', 'content': [{'type': 'text', 'text': GEMMA_SYSTEM_PROMPT}]},
        {'role': 'user', 'content': [
            {'type': 'image', 'image': image},
            {'type': 'text',  'text': question},
        ]},
    ]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors='pt',
        enable_thinking=False,
    ).to(model.device)
    return inputs.to(dtype=model.dtype)


def decode_response(processor, inputs, outputs):
    input_len = inputs['input_ids'].shape[-1]
    return processor.decode(outputs[0][input_len:], skip_special_tokens=True).strip()


def run_inference_timed(processor, model, family, image, question):
    if family == 'paligemma':
        inputs = build_paligemma_inputs(processor, image, question, model)
    elif family == 'gemma4':
        inputs = build_gemma_inputs(processor, image, question, model)
    else:
        raise ValueError(f'Unknown family {family!r}')

    _sync()
    t0 = time.time()
    gen_kwargs = dict(
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        num_beams=1,
        pad_token_id=processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id,
        eos_token_id=processor.tokenizer.eos_token_id,
    )
    with torch.no_grad():
        output = model.generate(**inputs, **gen_kwargs)
    _sync()
    latency_ms = (time.time() - t0) * 1000.0
    return decode_response(processor, inputs, output), latency_ms


print('Inference primitives defined')


## 9. Task list builder

Flattens the Experiment 2 questions over their applicable images into tasks.

In [ ]:
def build_task_list(questions, ground_truth, images_path):
    tasks = []
    for q in questions:
        applies = set(q['applies_to_categories'])
        applies_field = q.get('applies_only_when_field_equals', None)
        for img_rel, gt in ground_truth.items():
            if gt['category'] not in applies:
                continue
            if applies_field is not None:
                ok = all(gt.get(k) == v for k, v in applies_field.items())
                if not ok:
                    continue
            expected = gt.get(q['expected_answer_field'])
            if expected is None:
                continue
            full_path = os.path.join(images_path, img_rel)
            if not os.path.isfile(full_path):
                print(f'  WARNING: missing image {full_path}')
                continue
            tasks.append({
                'image_rel': img_rel,
                'image_path': full_path,
                'category': gt['category'],
                'question_id': q['id'],
                'question_text': q['question_text'],
                'expected': expected,
                'task_type': q['task_type'],
                'ground_truth': gt,
            })
    return tasks


tasks_preview = build_task_list(exp2_questions, ground_truth, IMAGES_PATH)
print(f'Total Exp2 tasks: {len(tasks_preview)}')
by_q = defaultdict(int)
for t in tasks_preview:
    by_q[t['question_id']] += 1
for q_id, n in by_q.items():
    print(f'  {q_id}: {n} tasks')


## 10. Per-model evaluation pipeline

For one model: load, warm up, run 3 timed passes, compute the extraction metrics,
save the artefacts (`results.csv`, `aggregate.csv`, `efficiency.json`,
`matrices.json`), and free GPU memory.

In [ ]:
def get_question_def(q_id):
    for q in exp2_questions:
        if q['id'] == q_id:
            return q
    raise KeyError(q_id)


FLOOR_CLASSES = ['G 0', 'G 1', 'G 2', 'G Agora']


def compute_floor_extraction_metrics(df):
    """Metrics for floor_extraction question (4-class classification by parsing)."""
    sub = df[df['question_id'] == 'floor_extraction']
    if sub.empty:
        return {}

    out = {'question_id': 'floor_extraction', 'n': len(sub)}
    expected_full = sub['expected'].astype(str).tolist()
    raw_predictions = sub['prediction'].astype(str).tolist()

    # Parse predictions -> canonical class
    parsed_class = []
    parsed_building = []
    parsed_floor = []
    for p in raw_predictions:
        b, f = parse_directory_sign_response(p)
        parsed_building.append(b)
        parsed_floor.append(f)
        parsed_class.append(format_parsed(b, f))

    # Expected components
    expected_building = []
    expected_floor = []
    for e in expected_full:
        b, f = parse_directory_sign_response(e)
        expected_building.append(b)
        expected_floor.append(f)

    # Component accuracies
    out['floor_accuracy']    = round(sum(1 for e, p in zip(expected_floor, parsed_floor) if e == p) / len(sub), 4)
    out['building_accuracy'] = round(sum(1 for e, p in zip(expected_building, parsed_building) if e == p) / len(sub), 4)
    out['location_accuracy'] = round(sum(1 for ef, pf, eb, pb in zip(expected_floor, parsed_floor, expected_building, parsed_building) if ef == pf and eb == pb) / len(sub), 4)

    # Macro P/R/F1 over the 4 floor classes
    m = macro_prf(expected_full, parsed_class, FLOOR_CLASSES)
    out['macro_precision'] = m['macro_precision']
    out['macro_recall']    = m['macro_recall']
    out['macro_f1']        = m['macro_f1']
    out['classes_used']    = m['classes_used']

    # BertScore on raw responses
    out['bertscore_f1'] = compute_bertscore(raw_predictions, expected_full)

    # Avg output length
    out['avg_output_length_words'] = avg_output_length_words(raw_predictions)

    return out


def compute_room_code_metrics(df):
    """Metrics for room_code_extraction question (substring match)."""
    sub = df[df['question_id'] == 'room_code_extraction']
    if sub.empty:
        return {}

    out = {'question_id': 'room_code_extraction', 'n': len(sub)}
    n_correct = sum(1 for _, row in sub.iterrows()
                    if room_code_match(row['prediction'], row['expected']))
    out['room_code_accuracy'] = round(n_correct / len(sub), 4)
    out['avg_output_length_words'] = avg_output_length_words(sub['prediction'].tolist())
    return out


def compute_floor_confusion_matrix(df):
    """4x4 confusion matrix + per-class P/R/F1 for floor_extraction."""
    sub = df[df['question_id'] == 'floor_extraction']
    if sub.empty:
        return {}
    expected = sub['expected'].astype(str).tolist()
    raw_preds = sub['prediction'].astype(str).tolist()
    parsed = []
    for p in raw_preds:
        b, f = parse_directory_sign_response(p)
        parsed.append(format_parsed(b, f))

    cm = confusion_matrix_dict(expected, parsed, FLOOR_CLASSES)
    per_c = per_class_prf(expected, parsed, FLOOR_CLASSES)
    return {
        'classes': FLOOR_CLASSES,
        'matrix': cm,
        'per_class_metrics': per_c,
        'note': 'Rows = ground truth, columns = predicted. "unparseable" column collects predictions whose parsing failed.',
    }


print('Aggregator helpers defined')


In [ ]:
def run_experiment2_for_model(meta):
    model_id = meta['model_id']
    family = meta.get('family', 'paligemma')
    safe_name = model_id.replace('/', '_').replace('-', '_')
    model_dir = f'{RESULTS_PATH}/{safe_name}'
    os.makedirs(model_dir, exist_ok=True)

    print(f'\n{"=" * 60}')
    print(f'MODEL: {model_id}  (family={family})')
    print(f'{"=" * 60}')

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    t_load = time.time()
    processor, model, family = load_model(meta)
    model_load_time_s = round(time.time() - t_load, 2)
    n_parameters = sum(p.numel() for p in model.parameters())
    print(f'  Loaded on {DEVICE} - {n_parameters / 1e9:.2f}B params in {model_load_time_s:.1f}s')

    print(f'  Warming up ({N_WARMUP} iters on synthetic image, discarded)...')
    warmup_img = make_synthetic_image()
    for _ in range(N_WARMUP):
        run_inference_timed(processor, model, family, warmup_img, 'What is in this image?')

    tasks = build_task_list(exp2_questions, ground_truth, IMAGES_PATH)
    n_tasks = len(tasks)
    print(f'  Tasks to run: {n_tasks} per latency run x {N_LATENCY_RUNS} runs')

    predictions = [None] * n_tasks
    latencies = [[] for _ in range(n_tasks)]
    t_total = time.time()

    for run_idx in range(N_LATENCY_RUNS):
        order = np.arange(n_tasks)
        np.random.default_rng(SEED * 1000 + run_idx).shuffle(order)
        for task_idx in order:
            task = tasks[task_idx]
            image = Image.open(task['image_path']).convert('RGB')
            try:
                pred, lat_ms = run_inference_timed(processor, model, family, image, task['question_text'])
            except Exception as e:
                print(f'    [FAIL] run {run_idx + 1} task {task_idx}: {e}')
                pred, lat_ms = '', 0.0
            latencies[task_idx].append(lat_ms)
            if run_idx == 0:
                predictions[task_idx] = pred

    total_time_s = time.time() - t_total

    rows = []
    for t, pred, lats in zip(tasks, predictions, latencies):
        lat_mean = float(np.mean(lats)) if lats else 0.0
        lat_std = float(np.std(lats, ddof=1)) if len(lats) >= 2 else 0.0
        # Compute correctness for results.csv (different per question)
        if t['question_id'] == 'floor_extraction':
            eb, ef = parse_directory_sign_response(t['expected'])
            pb, pf = parse_directory_sign_response(pred)
            correct = (eb == pb and ef == pf)
        elif t['question_id'] == 'room_code_extraction':
            correct = room_code_match(pred, t['expected'])
        else:
            correct = False
        rows.append({
            'image': t['image_rel'],
            'category': t['category'],
            'question_id': t['question_id'],
            'expected': t['expected'],
            'prediction': pred,
            'correct': bool(correct),
            'inference_time_ms': round(lat_mean, 2),
            'latency_ms_std': round(lat_std, 2),
        })
        for i, lat in enumerate(lats):
            rows[-1][f'latency_ms_run{i}'] = round(lat, 2)
    df = pd.DataFrame(rows)

    print(f'  Inferences: {len(df)} unique tasks, {len(df) * N_LATENCY_RUNS} timed calls in {total_time_s:.1f}s')

    # Per-question metrics
    agg_rows = [compute_floor_extraction_metrics(df), compute_room_code_metrics(df)]
    agg_rows = [r for r in agg_rows if r]
    agg_df = pd.DataFrame(agg_rows)

    overall_acc = round(df['correct'].mean(), 4) if not df.empty else 0.0
    floor_cm = compute_floor_confusion_matrix(df)

    vram_peak = torch.cuda.max_memory_allocated() / (1024 ** 3) if torch.cuda.is_available() else 0.0
    total_inf_ms = df['inference_time_ms'].sum()
    avg_inf_ms = float(df['inference_time_ms'].mean()) if not df.empty else 0.0
    avg_inf_ms_std = float(df['latency_ms_std'].mean()) if not df.empty else 0.0

    efficiency = {
        'model_id': model_id,
        'family': family,
        'experiment': 'exp2_update',
        'n_parameters': n_parameters,
        'n_parameters_billions': round(n_parameters / 1e9, 2),
        'n_inferences': len(df),
        'overall_accuracy': overall_acc,
        'vram_peak_gb': round(vram_peak, 3),
        'model_load_time_s': model_load_time_s,
        'total_wall_time_s': round(total_time_s, 2),
        'mean_inference_time_ms': round(avg_inf_ms, 2),
        'avg_inference_time_ms_std': round(avg_inf_ms_std, 2),
        'throughput_inf_per_s': round(len(df) / (total_inf_ms / 1000), 3) if total_inf_ms > 0 else 0,
    }

    matrices_payload = {
        'model_id': model_id,
        'family': family,
        'experiment': 'exp2_update',
        'floor_extraction': floor_cm,
    }

    df.to_csv(f'{model_dir}/results.csv', index=False)
    agg_df.to_csv(f'{model_dir}/aggregate.csv', index=False)
    with open(f'{model_dir}/efficiency.json', 'w') as f:
        json.dump(efficiency, f, indent=2)
    with open(f'{model_dir}/matrices.json', 'w') as f:
        json.dump(matrices_payload, f, indent=2)

    print(f'  All files saved to {model_dir}')

    summary = {
        'model_id': model_id,
        'model_safe': safe_name,
        'family': family,
        'version': meta.get('version', ''),
        'size': meta.get('size', ''),
        'type': meta.get('type', ''),
        'resolution': meta.get('resolution', ''),
        'n_parameters_billions': round(n_parameters / 1e9, 2),
        'overall_accuracy': overall_acc,
        'vram_peak_gb': efficiency['vram_peak_gb'],
        'model_load_time_s': model_load_time_s,
        'mean_inference_time_ms': efficiency['mean_inference_time_ms'],
        'avg_inference_time_ms_std': efficiency['avg_inference_time_ms_std'],
        'throughput_inf_per_s': efficiency['throughput_inf_per_s'],
    }
    for _, row in agg_df.iterrows():
        q_id = row['question_id']
        for k in row.index:
            if k in ('question_id', 'classes_used'): continue
            v = row[k]
            if pd.notna(v):
                summary[f'{q_id}_{k}'] = v

    del model, processor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print('  GPU memory freed')
    return summary


print('Orchestrator defined')


## 11. Main loop

Filters `models.json` to the two finalists and runs each. A model that crashes is
recorded in `failed_models` instead of aborting the batch.

In [ ]:
# Filter to the two Exp1 finalists (top performer of each family)
EXP2_FINALISTS = ['google/paligemma2-3b-mix-448', 'google/gemma-4-E4B-it']
run_models = [m for m in run_models if m['model_id'] in EXP2_FINALISTS]
print(f'Exp2 evaluates {len(run_models)} finalists (selected from Exp1):')
for m in run_models:
    print(f'  - {m["model_id"]}')
print()

all_summaries = []
failed_models = []

for meta in run_models:
    model_id = meta['model_id']
    try:
        summary = run_experiment2_for_model(meta)
        all_summaries.append(summary)
        print(f'  OK: {model_id}  overall_accuracy={summary["overall_accuracy"]}')
    except Exception as exc:
        print(f'  FAILED: {model_id} - {exc}')
        failed_models.append({'model_id': model_id, 'error': str(exc)})
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print(f'\nCompleted: {len(all_summaries)}/{len(run_models)} models')
if failed_models:
    print('Failed:', [m['model_id'] for m in failed_models])
    with open(f'{RESULTS_PATH}/failed_models.json', 'w') as f:
        json.dump(failed_models, f, indent=2)


## 12. Cross-model comparison table

All metrics per model, saved to `all_models_summary.csv`.

In [ ]:
comp_df = pd.DataFrame(all_summaries)

headline_cols = [
    'model_id', 'family', 'version', 'size', 'type', 'resolution', 'n_parameters_billions',
    'overall_accuracy',
    # floor_extraction
    'floor_extraction_floor_accuracy',
    'floor_extraction_building_accuracy',
    'floor_extraction_location_accuracy',
    'floor_extraction_macro_precision',
    'floor_extraction_macro_recall',
    'floor_extraction_macro_f1',
    'floor_extraction_bertscore_f1',
    'floor_extraction_avg_output_length_words',
    # room_code_extraction
    'room_code_extraction_room_code_accuracy',
    'room_code_extraction_avg_output_length_words',
    # operacional
    'vram_peak_gb', 'model_load_time_s',
    'mean_inference_time_ms', 'avg_inference_time_ms_std', 'throughput_inf_per_s',
]
headline_cols = [c for c in headline_cols if c in comp_df.columns]
summary_df = comp_df[headline_cols].copy()

summary_df.to_csv(f'{RESULTS_PATH}/all_models_summary.csv', index=False)
print(f'Saved: {RESULTS_PATH}/all_models_summary.csv')
print()
print('ALL MODELS - SUMMARY')
print('=' * 80)
display(summary_df.round(4))


## 13. Console summary

The comparison table is already saved in section 12. This cell prints the best
model per headline metric.

In [ ]:
if not comp_df.empty:
    print('=== EXP2 - BEST MODEL PER METRIC ===')
    for metric in ['overall_accuracy',
                   'floor_extraction_location_accuracy',
                   'floor_extraction_floor_accuracy',
                   'floor_extraction_macro_f1',
                   'floor_extraction_bertscore_f1',
                   'room_code_extraction_room_code_accuracy']:
        if metric in comp_df.columns:
            s = comp_df[metric].dropna()
            if not s.empty:
                idx = s.idxmax()
                print(f'  {metric}: {comp_df.loc[idx, "model_id"]} ({s.max():.4f})')
    print()
    print('=== FASTEST (lowest mean_inference_time_ms) ===')
    s = comp_df['mean_inference_time_ms'].dropna()
    if not s.empty:
        idx = s.idxmin()
        print(f'  {comp_df.loc[idx, "model_id"]} ({s.min():.1f} ms)')
else:
    print('No models completed successfully.')


## 14. Download results as ZIP

Packages everything under the results path into a single ZIP and triggers a browser
download.

In [ ]:
import shutil
from google.colab import files

zip_base = '/content/exp2_update_results'
shutil.make_archive(zip_base, 'zip', RESULTS_PATH)
zip_path = zip_base + '.zip'
print(f'Created {zip_path}  ({os.path.getsize(zip_path) / 1024**2:.2f} MB)')
files.download(zip_path)
